In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Install dependencies
!pip install -q transformers accelerate qwen_vl_utils bitsandbytes streamlit pillow
# Install localtunnel to expose the app to your laptop
#!npm install -g localtunnel

In [ ]:
# Run this in a Kaggle cell
!pip install -q  openpyxl

In [1]:
%%writefile app.py
import streamlit as st
import torch
import os
import zipfile
import shutil
import pandas as pd
import time
from tqdm import tqdm
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

# --- CONFIG & DIRECTORIES ---
st.set_page_config(layout="wide", page_title="GeoData Excel Extractor")
st.title("🗺️ GeoTag Data to Excel Extractor (V3)")

# Unique temp folder per session to avoid mixing data
if 'session_id' not in st.session_state:
    st.session_state['session_id'] = str(int(time.time()))

SESSION_UPLOAD = f"uploads_{st.session_state['session_id']}"
if not os.path.exists(SESSION_UPLOAD): os.makedirs(SESSION_UPLOAD)

# --- MODEL LOADING (Standardized for T4) ---
@st.cache_resource
def load_qwen():
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True, 
        bnb_4bit_compute_dtype=torch.float16
    )
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        "Qwen/Qwen2.5-VL-7B-Instruct", 
        quantization_config=quant_config, 
        device_map="auto"
    )
    processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")
    return model, processor

# --- HYBRID UPLOADER (ZIP + Individual) ---
st.header("1. Upload Images")
files = st.file_uploader("Upload Images (JPEG/PNG) or a single ZIP file", 
                         type=["zip", "jpg", "jpeg", "png"], 
                         accept_multiple_files=True)

all_images = []

if files:
    with st.spinner("Processing uploads..."):
        for f in files:
            if f.name.endswith(".zip"):
                with zipfile.ZipFile(f, 'r') as z:
                    # Unzip and filter for standard images
                    z.extractall(SESSION_UPLOAD)
                    temp_files = [i for i in z.namelist() if i.lower().endswith(('.jpg', '.jpeg', '.png'))]
                    # Handle zipped folders properly
                    for tf in temp_files:
                        src = os.path.join(SESSION_UPLOAD, tf)
                        dst = os.path.join(SESSION_UPLOAD, os.path.basename(tf))
                        if src != dst:
                            shutil.move(src, dst)
            else:
                with open(os.path.join(SESSION_UPLOAD, f.name), "wb") as buf:
                    buf.write(f.getbuffer())

    # Final list of extracted images
    all_images = [i for i in os.listdir(SESSION_UPLOAD) if i.lower().endswith(('.jpg', '.jpeg', '.png'))]
    all_images.sort() # for consistency
    st.success(f"Found {len(all_images)} images ready for extraction.")

# --- EXTRACTION ENGINE ---
if all_images and st.button("2. 🔍 Run Qwen OCR Extraction"):
    model, processor = load_qwen()
    
    results = []
    
    # Setup Progress Bar
    prog_bar = st.progress(0)
    img_placeholder = st.empty() # Show image being processed
    
    # Prompt optimized to ensure Date is captured
    prompt = (
        "Look at the dark GPS Map Camera watermark at the bottom. "
        "Find the specific lines of text. DO NOT miss the Date line. "
        "Return the data EXACTLY as: Address Header | Latitude | Longitude | Date | Time | Full Details. "
        "Use '|' as separator. If missing, write 'N/A'."
    )
    
    for idx, filename in enumerate(tqdm(all_images)):
        img_path = os.path.join(SESSION_UPLOAD, filename)
        
        # UI updates
        img_placeholder.image(img_path, caption=f"Processing {filename}...", width=300)
        prog_bar.progress((idx + 1) / len(all_images), text=f"Processing image {idx+1}/{len(all_images)}")

        # AI Execution
        messages = [{"role": "user", "content": [
            {"type": "image", "image": img_path},
            {"type": "text", "text": prompt}
        ]}]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, _ = process_vision_info(messages)
        inputs = processor(text=[text], images=image_inputs, return_tensors="pt").to("cuda")
        
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=256)
            raw_output = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
            clean_output = raw_output.split("assistant\n")[-1].strip()

        # Data Parsing
        parts = [p.strip() for p in clean_output.split("|")]
        # Pad if Qwen didn't follow formatting
        while len(parts) < 6:
            parts.append("N/A")

        results.append({
            "File Name": filename,
            "Address (Header)": parts[0],
            "Lat (Raw)": parts[1],
            "Long (Raw)": parts[2],
            "Date (OCR)": parts[3],
            "Time (OCR)": parts[4],
            "Full Details String": parts[5]
        })
        
        # Clear VRAM for stability
        del inputs, generated_ids
        torch.cuda.empty_cache()

    # Clear UI placeholders
    img_placeholder.empty()
    st.success("Extraction Complete!")

    # --- DATAFRAME & EXCEL EXPORT ---
    st.header("3. Results Table")
    df = pd.DataFrame(results)
    st.dataframe(df, use_container_width=True)

    # Convert DataFrame to Excel bytes for Streamlit
    with st.spinner("Creating Excel file..."):
        excel_name = f"geodata_report_{st.session_state['session_id']}.xlsx"
        # Exporting to a buffer instead of a file on disk for direct download
        import io
        buffer = io.BytesIO()
        df.to_excel(buffer, index=False, engine='openpyxl')
        buffer.seek(0)

    st.download_button(
        label=f"📥 Download Excel Report (.xlsx)",
        data=buffer,
        file_name=excel_name,
        mime="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
    )

# Cleanup logic for old session folders (run on script start)
# [Optional: in production, add logic to delete folders > 1hr old]

Overwriting app.py


In [ ]:
!pip install pyngrok

from pyngrok import ngrok
# Replace 'YOUR_TOKEN' with the token from ngrok.com
ngrok.set_auth_token("3CkI2vKqoLWRE55Bi9a5sIVIzku_7WHGyGtLkXKDXBoYeJPLv")

# Open the tunnel
public_url = ngrok.connect(8501)
print(f"Click here to open your App on your laptop: {public_url}")


In [2]:
import os
import subprocess
import threading
import time
import socket

# 1. Kill old "zombie" processes to free up port 8501
!fuser -k 8501/tcp
!pkill lt
!pkill ngrok

# 2. Start Streamlit and Force IPv4
def run_app():
    # We force the address to 127.0.0.1 and send errors to a log file
    with open("streamlit_output.log", "w") as f:
        subprocess.run([
            "streamlit", "run", "app.py", 
            "--server.port", "8501", 
            "--server.address", "127.0.0.1",
            "--server.headless", "true"
        ], stdout=f, stderr=f)

# Start the thread
thread = threading.Thread(target=run_app, daemon=True)
thread.start()

# 3. Health Check Loop
print("⏳ Loading Qwen2.5-VL (This takes ~2-3 minutes for the first run)...")
for i in range(60): # Try for 5 minutes
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    result = sock.connect_ex(('127.0.0.1', 8501))
    if result == 0:
        print("\n✅ Streamlit is UP and LISTENING!")
        sock.close()
        break
    else:
        # Check if the log file shows a crash
        if os.path.exists("streamlit_output.log"):
            with open("streamlit_output.log", "r") as f:
                content = f.read()
                if "Error" in content or "Traceback" in content:
                    print("\n❌ STREAMLIT CRASHED! See error below:")
                    print(content[-500:]) # Show last 500 chars of error
                    break
        print(".", end="", flush=True)
        time.sleep(5)
    sock.close()

# 4. Connect Ngrok ONLY after the app is confirmed live
from pyngrok import ngrok
# Ensure your token is set here if you haven't run it yet:
# ngrok.set_auth_token("YOUR_TOKEN_HERE")

public_url = ngrok.connect("127.0.0.1:8501", bind_tls=True)
print(f"\n🔗 Access your app on your laptop here: {public_url}")

⏳ Loading Qwen2.5-VL (This takes ~2-3 minutes for the first run)...
.
✅ Streamlit is UP and LISTENING!

🔗 Access your app on your laptop here: NgrokTunnel: "https://football-treadmill-confess.ngrok-free.dev" -> "http://127.0.0.1:8501"
